# FeCAM full vs identity ablation (CIFAR-100)

This notebook trains the base backbone once, then evaluates `full` and `identity` with that same checkpoint on GPU 0 and GPU 1. It writes per-class results and one overall summary; base/incremental group accuracy can be derived from the per-class table. The full run uses the repository reproduction configs (100 classes, B=50, increment=10, 200 epochs).

In [ ]:
from pathlib import Path
import os, subprocess, sys, time

REPO_URL = 'https://github.com/duongthienphu/fecam-study.git'
REPO_BRANCH = 'feat/evaluation-tracking'
REPO_DIR = Path('/kaggle/working/fecam-study')
if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets==4.8.5', 'PyYAML==6.0.3', 'psutil==7.2.2'], check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR, '| branch:', REPO_BRANCH)

In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('In Kaggle Settings select GPU T4 x2, then restart the session.')
print('GPU 0:', torch.cuda.get_device_name(0))
print('GPU 1:', torch.cuda.get_device_name(1))

## 1. Train one shared base backbone

This cell trains only if the shared checkpoint does not exist. Keep the Kaggle working directory for the later evaluation cells.

In [ ]:
BASE_DIR = REPO_DIR / 'outputs' / 'fecam-mode-shared-base'
BASE_CKPT = BASE_DIR / 'base.pt'
if BASE_CKPT.is_file():
    print('Using existing shared checkpoint:', BASE_CKPT)
else:
    if BASE_DIR.exists():
        raise RuntimeError(f'{BASE_DIR} exists without base.pt; use a fresh Kaggle working directory or a new BASE_DIR.')
    command = [sys.executable, 'scripts/run_experiment.py', 'train-base',
               '--config', 'configs/reproduction/cifar100_fecam.yaml',
               '--device', 'cuda:0', '--output', str(BASE_DIR)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
assert BASE_CKPT.is_file(), BASE_CKPT
print('Shared base checkpoint ready:', BASE_CKPT)

## 2. Evaluate both modes on the same checkpoint

Both evaluations use identical protocol, seed, dataset, class order, training config, and checkpoint. Only `classifier.mode` differs.

In [ ]:
import json
RUNS = {
    'full': {'config': 'configs/reproduction/cifar100_fecam.yaml', 'gpu': 0},
    'identity': {'config': 'configs/reproduction/cifar100_fecam_identity.yaml', 'gpu': 1},
}
jobs = {}
child_env = os.environ.copy()
child_env['OMP_NUM_THREADS'] = '2'
child_env['MKL_NUM_THREADS'] = '2'
for mode, spec in RUNS.items():
    out = REPO_DIR / 'outputs' / f'fecam-mode-{mode}'
    summary = out / 'summary.json'
    if summary.is_file():
        print(mode, 'already complete; keeping', summary)
        continue
    if out.exists():
        raise RuntimeError(f'{out} exists without summary.json; use a fresh output name before retrying.')
    log = Path('/kaggle/working') / f'fecam-mode-{mode}.log'
    handle = log.open('w', encoding='utf-8')
    command = [sys.executable, 'scripts/run_experiment.py', 'evaluate',
               '--config', spec['config'], '--checkpoint', str(BASE_CKPT),
               '--device', f"cuda:{spec['gpu']}", '--output', str(out)]
    proc = subprocess.Popen(command, cwd=REPO_DIR, env=child_env, stdout=handle, stderr=subprocess.STDOUT)
    jobs[mode] = {'proc': proc, 'handle': handle, 'log': log}
    print('Started', mode, 'on cuda:' + str(spec['gpu']), '| log:', log)

failed = []
for mode, job in jobs.items():
    code = job['proc'].wait()
    job['handle'].close()
    if code:
        failed.append(mode)
        print(mode, 'failed; log tail:\n', job['log'].read_text(encoding='utf-8', errors='replace')[-8000:])
    else:
        print(mode, 'completed')
if failed:
    raise RuntimeError('Evaluation failed: ' + ', '.join(failed))

## 3. Compare per-class and group accuracy

`base` is the initial 50 classes. `incremental` means all classes introduced after task 0 and seen by that session. Deltas are `identity - full` in percentage points.

In [ ]:
COMPARE_DIR = REPO_DIR / 'outputs' / 'fecam-mode-comparison'
subprocess.run([sys.executable, 'scripts/compare_fecam_modes.py',
                '--full-summary', 'outputs/fecam-mode-full/summary.json',
                '--identity-summary', 'outputs/fecam-mode-identity/summary.json',
                '--output-dir', str(COMPARE_DIR)], cwd=REPO_DIR, check=True)
import pandas as pd
display(pd.read_csv(COMPARE_DIR / 'fecam_mode_per_class_comparison.csv'))
display(pd.read_csv(COMPARE_DIR / 'fecam_mode_summary_comparison.csv'))